# الدرس السابع: الإخراج المهيكل باستخدام TypedDict (Structured Output with TypedDict)

## المقدمة والاهداف التعليمية
في هذا الدفتر، سنتعلم كيفية استخدام `typing.TypedDict` مع الدالة `with_structured_output` كبديل خفيف الوزن (Lightweight) لـ Pydantic.

## متى تفضل استخدام `TypedDict` على `Pydantic`؟
1. عندما ترغب في تجنب الاعتماد على مكتبات خارجية، واستخدام ما توفره مكتبة بايثون القياسية (`typing`).
2. عندما تحتاج الى معالجة قواميس بايثون نقية (`dict`) مباشرة دون الحاجة الى دوال التحويل الخاصة بكائنات الفئات.
3. في التطبيقات التي تتطلب اقصى سرعة في زمن التنفيذ ولا تحتاج الى قواعد تحقق معقدة على مستوى الحقول.

## الخطوة 1: تهيئة البيئة واستيراد TypedDict و Annotated
نستخدم `typing.Annotated` لاضافة اوصاف الحقول التوضيحية التي توجه نموذج اللغة اثناء تعبئة البيانات.

In [ ]:
import os
from typing import TypedDict, Annotated, List, Optional
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq", temperature=0)

## الخطوة 2: تعريف مخطط البيانات باستخدام TypedDict
نعرف بنية لتقييم المخاطر الامنية للمكالمات او الرسائل الاحتيالية.

In [ ]:
class SecurityIncidentReport(TypedDict):
    threat_level: Annotated[str, "Level of severity: Low, Medium, High, or Critical"]
    incident_type: Annotated[str, "Category such as Phishing, Malware, Unauthorized Access, or Social Engineering"]
    affected_systems: Annotated[List[str], "Names or types of affected infrastructure and services"]
    recommended_action: Annotated[str, "Immediate defensive mitigation step recommended"]
    requires_escalation: Annotated[bool, "True if human executive intervention is immediately required"]

print("TypedDict Schema successfully created.")

## الخطوة 3: استدعاء النموذج والحصول على قاموس مهيكل
نمرر الفئة `SecurityIncidentReport` مباشرة الى `with_structured_output`.

In [ ]:
typed_llm = model.with_structured_output(SecurityIncidentReport)

incident_log = """
Urgent Alert: An employee received an email impersonating IT Support with a suspicious link.
The employee clicked and entered their credentials for the internal CRM and database portal.
We detected unusual external queries originating from their account.
Immediate action taken: Credentials revoked, sessions invalidated.
SOC team needs to isolate the CRM host immediately and notify leadership.
"""

# استخراج التقرير المهيكل
report_dict = typed_llm.invoke(incident_log)

print("Result Type:", type(report_dict))
print("Threat Level:", report_dict["threat_level"])
print("Incident Type:", report_dict["incident_type"])
print("Affected Systems:", report_dict["affected_systems"])
print("Recommended Action:", report_dict["recommended_action"])
print("Requires Escalation:", report_dict["requires_escalation"])

## الخطوة 4: المقارنة البرمجية بين TypedDict و Pydantic
نلخص الفرق البرمجي بين الاسلوبين:
- كائنات Pydantic تتيح الوصول للحقول عبر النقطة (`obj.field`) والتحقق الصارم من صحة النوع (Type validation).
- كائنات TypedDict تعيد قواميس بايثون عادية يتم الوصول اليها عبر المفاتيح (`obj['key']`) وتكون سريعة جدا وخفيفة في الذاكرة.

In [ ]:
print("Verification of Dictionary Properties:")
print("Keys available:", list(report_dict.keys()))
print("Direct dictionary access test:", report_dict.get("threat_level"))